### Play Tic-tac-toe against an agent trained with Reinforcement Learning

Where we don't hardcode the decision the computer should make, but train an agent for it instead.

This is an example of Reinforement Learning. Instead of training a model on some dataset, which is (un)supervised learning, we essentially set up a number of rules with which the algorithm can earn "points" if it is playing the game correctly.

To start, we'll define the game itself and an Agent class to play it, which for now will just select random positions to place its mark.

In [2]:
import numpy as np
from pathlib import Path
import pickle

In [3]:
# Start by defining the board state
# A given spot is either a space for nothing, or an X or O mark
class Board():
    def __init__(self):
        self.board = [" ", " ", " ",
                      " ", " ", " ",
                      " ", " ", " "]

    def display(self):
        print(self.board[0] + " | " + self.board[1] + " | " + self.board[2])
        print("-- --- --")
        print(self.board[3] + " | " + self.board[4] + " | " + self.board[5])
        print("-- --- --")
        print(self.board[6] + " | " + self.board[7] + " | " + self.board[8])

    def place(self, mark: str, position: int):
        if mark not in ["X", "O"]:
            raise ValueError("Mark must be X or O")
        if self.board[position] != " ":
            raise ValueError("Position is not empty")
        self.board[position] = mark

board = Board()
board.display()
print()
board.place("X", 3)
board.place("O", 8)
board.display()

  |   |  
-- --- --
  |   |  
-- --- --
  |   |  

  |   |  
-- --- --
X |   |  
-- --- --
  |   | O


In [4]:
class Agent:
    def __init__(self):
        pass

    def place_mark(self, board_state: Board) -> int:
        # Just select a random position for now
        empty_indexes = np.where(np.array(board_state.board, dtype=str) == " ")[0]
        return np.random.choice(empty_indexes)

# Define the game itself
class Game:
    def __init__(self):
        self.board_state = Board()

    def ask_mark_position(self) -> int:
        while True:
            mark_index = input("Choose in which position (1-9) to place your mark: ")
            if not mark_index.isdigit():
                print("Please enter an integer.")
                continue
            
            mark_index = int(mark_index)
            if mark_index < 1 or mark_index > 9:
                print("Please enter a number between 1 and 9.")
            elif self.board_state.board[mark_index - 1] != " ":
                print("Please choose an empty position.")
            else:
                break
        return mark_index - 1

    def check_game_over(self, mark: str) -> bool:
        win_combinations = [
            (0,1,2), (3,4,5), (6,7,8),
            (0,3,6), (1,4,7), (2,5,8),
            (0,4,8), (2,4,6)
        ]
        mark_indexes = np.where(np.array(self.board_state.board, dtype=str) == mark)[0]
        for combination in win_combinations:
            if combination[0] in mark_indexes and combination[1] in mark_indexes and combination[2] in mark_indexes:
                return True
        return False

    # Play with two players in the terminal
    def play(self):
        turns = 0
        while turns < 9:
            self.board_state.display()
            if turns % 2 == 0:
                print("X's turn")
                self.board_state.place("X", self.ask_mark_position())
                if self.check_game_over("X"):
                    print()
                    print("X wins!")
                    self.board_state.display()
                    break
            else:
                print("O's turn")
                self.board_state.place("O", self.ask_mark_position())
                if self.check_game_over("O"):
                    print()
                    print("O wins!")
                    self.board_state.display()
                    break
            turns += 1
            print()

        if turns == 9:
            print("Draw!")
            self.board_state.display()

    # Play against the agent, where you are X, and the agent is O
    def play_with_agent(self, agent: Agent):
        turns = 0
        while turns < 9:
            self.board_state.display()
            if turns % 2 == 0:
                print("X's turn")
                self.board_state.place("X", self.ask_mark_position())
                if self.check_game_over("X"):
                    print()
                    print("X wins!")
                    self.board_state.display()
                    break
            else:
                print("O's turn")
                placement_index = agent.place_mark(self.board_state)
                print(f"Agent places O at position {placement_index + 1}")
                self.board_state.place("O", placement_index)
                if self.check_game_over("O"):
                    print()
                    print("O wins!")
                    self.board_state.display()
                    break
            turns += 1
            print()

        if turns == 9:
            print("Draw!")
            self.board_state.display()

Now let's look at the theory of Reinforcement Learning in more detail. As stated above, we will be training an agent to make the correct decisions at any point during the game, without hardcoding exactly what it should do. The training will involve the agent trying out many different actions it can take during a game when it is in a given state, and us rewarding it when it makes a good decision, and penalizing it for a bad decision. This information is stored in the so-called Q-table; the goal of training is to fill up this table. This table has an entry for every possible state of the game and every action that can be taken at that point. It is denoted as $Q(s,a)$, where $s$ is the given state and $a$ the given action.

In our case of Tic-tac-toe, the state is given by where on the board X's and O's are placed, and the action is placing a mark in one of the remaining empty positions, which can be represented by the positions' index. The Q-table is initially filled with zeros, and filled in based on the reward for every action in a given state. By the end of the training, it then contains the total expected reward for each action in each state.

To actually store the Q-table, it is easiest to use a dictionary, where the keys are the board states (which are lists) and the values are a list where each index corresponds to a position, so with length 9 (remember that `Q[i][j]` should give you the reward for action `j` at state `i`). This way board states can be appended as we come across them, which is easier than finding a mapping from each board state to a unique index as would be required for a 2D array. Lists cannot be used as dictionary keys though, so we will first turn the board state into a string that simply combines all the marks and spaces at each position.

In [5]:
Q = {}
# Example
state = ["O", "X", " ",
         " ", "O", " ",
         "X", " ", "X"]
state_str = "".join(state)
print(state_str)
Q[state_str] = [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
print(Q[state_str][5])

Q = {}

OX  O X X
1.0


Now there are a number of parameters to be set. Firstly, the rewards for playing well. In our case, we simply give a point if the agent wins the game, and subtract one if it loses. In case of a draw, nothing is added. This way the agent should tend towards winning and away from losing, without us actually telling it what strategy to use. Any intermediate step then also does not offer a reward; this could also be penalized to make the model finish games faster, but that is not relevant in this case.

In [6]:
reward_win = 1
reward_draw = 0
reward_loss = -1
reward_move = 0

So how do we then actually use these rewards train the model, i.e. fill up the Q-table? We use the following equation to update values:

\begin{equation}
\nonumber
    Q(s,a) = Q(s,a) + \alpha \left[ r + \gamma \left. \max(Q(s', a'))\right|_{a'} - Q(s,a) \right],
\end{equation}

where $Q(s,a)$ on the LHS is the updated value while on the RHS it is the old value, $\alpha$ is the learning rate, $r$ is the reward after taking action $a$ as defined above, $\gamma$ is the discount factor, and $s'$ is the state of the game after taking action $a$. The term $\left. \max(Q(s', a'))\right|_{a'}$ takes the maximum value currently in the table for the state $s'$ between all possible actions $a'$.

The function of the learning rate $\alpha$ is to control how much new results override old ones. By making it small, random wins will not skew the Q-table too much, as only consistent results can add up to more significant values. The point of the discount factor $\gamma$ is to determine how much future rewards matter compared to immediate ones, because it is multiplied with the $\left. \max(Q(s', a'))\right|_{a'}$ term, so the highest reward that you can get for any action in state $s'$ (meaning you play optimally). This value encodes the best future reward as well due to the recursive nature of the Q-table equation: a high future value of $Q(s')$ increases $Q(s)$ itself.

In [7]:
alpha = 0.1
gamma = 0.9

# For the agent class
def Q_update(Q: dict, state: list, action: int, reward: int, next_state: list) -> dict:
    state_str = "".join(state)
    next_state_str = "".join(next_state)

    # Check if state and next_state already exist, append zeros if not
    if state_str not in Q:
        Q[state_str] = [0.0]*9
    if next_state_str not in Q:
        Q[next_state_str] = [0.0]*9

    # Actual update
    Q[state_str][action] = Q[state_str][action] + alpha * (reward + gamma * np.max(Q[next_state_str]) - Q[state_str][action])
    return Q

There are two more parameters to set for the training. Firstly there is the number of times we will let the agent play the game during training; a larger value means a better trained model, but also increases training time. Secondly there is the $\epsilon$ parameter, which is the probability that the agent _does not_ pick its best known move for a given state, but rather chooses randomly. This is done to keep the agent from getting stuck in 'local maxima', meaning actions that lead to good results but not the very best results overal. The randomness leads to new situations that can potentially lead to better actions in the end. We start with a high probability for $\epsilon$, so a lot of random picks, and decrease the value during training; this way the agent initially explores many options, but can train the better ones near the end.

Additionally, we need to make sure the agent only gets to see empty positions, so that it does not end up returning invalid moves while it is still learning. There is also en extra modification for the case that values in the Q-table are tied, which happens especially often when all values are still zero. In that case we don't just want to return the first highest values move, as this creates a bias. So we will pick a move randomly between tied values instead.

In [ ]:
num_episodes = 1000
epsilon_start = 0.9  # to start
epsilon_end = 0.1  # at the end

def place_mark(board_state: Board, Q: dict, epsilon: float) -> int:
    # Epsilon probability for random move, also if state not encountered yet
    state_str = "".join(board_state.board)
    empty_indices = np.where(np.array(board_state.board, dtype=str) == " ")[0]
    if np.random.random() < epsilon or state_str not in Q:
        return np.random.choice(empty_indices)
    else:
        # Make sure that returned action index is of an empty position
        placement_index = empty_indices[0]
        best_Q_value = Q[state_str][placement_index]
        # Need this in case of ties in Q (e.g. multiple zeros),
        # to select randomly instead of the first
        best_Q_values_tied = [best_Q_value]
        indices_tied = [placement_index]

        for index in empty_indices[1:]:
            Q_value = Q[state_str][index]
            if Q_value > best_Q_value:
                placement_index = index
                best_Q_value = Q_value
                # Reset tied lists
                best_Q_values_tied = [best_Q_value]
                indices_tied = [placement_index]
            elif Q_value == best_Q_value:
                best_Q_values_tied.append(Q_value)
                indices_tied.append(index)

        if len(best_Q_values_tied) == 1:
            # Just return the corresponding index
            return int(placement_index)
        else:
            # Randomly select from tied indices
            return np.random.choice(indices_tied)

# Example
Q_ex = {}
state = ["O", "X", " ",
         " ", "O", " ",
         "X", " ", "X"]
state_str = "".join(state)
print(state_str)
Q_ex[state_str] = [0.0, 0.0, 0.0, 0.0, 2.0, 1.0, 0.0, 0.0, 0.0]
print(Q_ex[state_str][4])

board = Board()
board.board = state
#board.display()
print(place_mark(board, Q_ex, 0.0))  # 4 without randomness

OX  O X X
2.0
4


We can now implement the actual training algorithm. One important detail is that this game needs two players, and so the agent cannot train "on its own". Since we also don't want to play thousands of games ourselves, we need some automatic opponent. For now, we will just use a random choice as the opponent's move. (Alternatively, another agent could be used, but that makes things more complicated.)

One practical thing to note is that in this case, the 'state' and 'next state' we use to calculate the Q-table are not the state before and after the agent places a mark, respectively, but rather the state before the agent moves and the state after the opponent has placed another mark after the agent's move; both need to be input states that the agent reacts to, not the output state after the agent moved. Otherwise, the states would not be connected through the Q-table equation and no actual learning happens. The only exception to this occurs when the agent wins: there we need to add the 'winning state', after the agent moved, as the final 'next state' so that the agent gets rewarded for that last move. This causes no problems with connecting states since it is always the last one.

We then also add a function to the `Game` class where the agent plays against a random opponent on the X side, so that this is automated. This is run without any print statements, as these are no longer useful for thousands of training rounds.

In [ ]:
class Agent:
    def __init__(
        self,
        reward_win: int = 1,
        reward_draw: int = 0,
        reward_loss: int = -1,
        reward_move: int = 0,
        alpha: float = 0.1,
        gamma: float = 0.9,
        num_episodes: int = 5000,
        epsilon_start: float = 0.9,
        epsilon_end: float = 0.1
    ):
        self.reward_win = reward_win
        self.reward_draw = reward_draw
        self.reward_loss = reward_loss
        self.reward_move = reward_move
        self.alpha = alpha
        self.gamma = gamma
        self.num_episodes = num_episodes
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        # Get the Q-table from a file if it exists,
        # start training if not
        self.filename = "Q_table.pkl"
        Q_path = Path(self.filename)
        if Q_path.exists():
            with open(self.filename, 'rb') as f:
                self.Q = pickle.load(f)
        else:
            self.Q = {}
    
    def place_mark_random(self, board_state: Board) -> int:
        # For the random opponent
        empty_indexes = np.where(np.array(board_state.board, dtype=str) == " ")[0]
        return np.random.choice(empty_indexes)

    def place_mark(self, board_state: Board, epsilon: float) -> int:
        # Epsilon probability for random move, also if state not encountered yet
        state_str = "".join(board_state.board)
        empty_indices = np.where(np.array(board_state.board, dtype=str) == " ")[0]
        if np.random.random() < epsilon or state_str not in self.Q:
            return np.random.choice(empty_indices)
        else:
            # Make sure that returned action index is of an empty position
            placement_index = empty_indices[0]
            best_Q_value = self.Q[state_str][placement_index]
            # Need this in case of ties in Q (e.g. multiple zeros),
            # to select randomly instead of the first
            best_Q_values_tied = [best_Q_value]
            indices_tied = [placement_index]

            for index in empty_indices[1:]:
                Q_value = self.Q[state_str][index]
                if Q_value > best_Q_value:
                    placement_index = index
                    best_Q_value = Q_value
                    # Reset tied lists
                    best_Q_values_tied = [best_Q_value]
                    indices_tied = [placement_index]
                elif Q_value == best_Q_value:
                    best_Q_values_tied.append(Q_value)
                    indices_tied.append(index)

            if len(best_Q_values_tied) == 1:
                # Just return the corresponding index
                return int(placement_index)
            else:
                # Randomly select from tied indices
                return np.random.choice(indices_tied)

    def Q_update(self, state: list, action: int, reward: int, next_state: list):
        state_str = "".join(state)
        next_state_str = "".join(next_state)

        # Check if state and next_state already exist, append zeros if not
        if state_str not in self.Q:
            self.Q[state_str] = [0.0]*9
        if next_state_str not in self.Q:
            self.Q[next_state_str] = [0.0]*9

        # Actual update
        self.Q[state_str][action] = (
            self.Q[state_str][action] + self.alpha *
            (reward + self.gamma * np.max(self.Q[next_state_str]) - self.Q[state_str][action])
        )

    def save_Q_table(self):
        with open(self.filename, 'wb') as f:
            pickle.dump(self.Q, f)

    def train(self):
        for episode in range(self.num_episodes):
            # Pick linear or exponential epsilon decay
            #epsilon = self.epsilon_start - (self.epsilon_start - self.epsilon_end) * (episode / self.num_episodes)
            epsilon = self.epsilon_end + (self.epsilon_start - self.epsilon_end) * np.exp(-episode/(0.1*self.num_episodes))
            
            # Play the game, initialized to a new board each iteration
            game = Game()
            game.play_for_training(self, epsilon)

        # Save the filled Q-table
        self.save_Q_table()

    def run(self):
        # Check if the agent is trained
        if not self.Q:
            print("Commencing agent training")
            self.train()
            print("Training complete")

        # Play the game
        print("Now starting a game")
        game = Game()
        game.play_with_agent(self)

In [13]:
# Define the game itself with the training game function
class Game:
    def __init__(self):
        self.board_state = Board()

    def ask_mark_position(self) -> int:
        while True:
            mark_index = input("Choose in which position (1-9) to place your mark: ")
            if not mark_index.isdigit():
                print("Please enter an integer.")
                continue
            
            mark_index = int(mark_index)
            if mark_index < 1 or mark_index > 9:
                print("Please enter a number between 1 and 9.")
            elif self.board_state.board[mark_index - 1] != " ":
                print("Please choose an empty position.")
            else:
                break
        return mark_index - 1

    def check_game_over(self, mark: str) -> bool:
        win_combinations = [
            (0,1,2), (3,4,5), (6,7,8),
            (0,3,6), (1,4,7), (2,5,8),
            (0,4,8), (2,4,6)
        ]
        mark_indexes = np.where(np.array(self.board_state.board, dtype=str) == mark)[0]
        for combination in win_combinations:
            if combination[0] in mark_indexes and combination[1] in mark_indexes and combination[2] in mark_indexes:
                return True
        return False

    # Play with two players in the terminal
    def play(self):
        turns = 0
        while turns < 9:
            self.board_state.display()
            if turns % 2 == 0:
                print("X's turn")
                self.board_state.place("X", self.ask_mark_position())
                if self.check_game_over("X"):
                    print()
                    print("X wins!")
                    self.board_state.display()
                    break
            else:
                print("O's turn")
                self.board_state.place("O", self.ask_mark_position())
                if self.check_game_over("O"):
                    print()
                    print("O wins!")
                    self.board_state.display()
                    break
            turns += 1
            print()

        if turns == 9:
            print("Draw!")
            self.board_state.display()

    # Play against the agent, where you are X, and the agent is O
    def play_with_agent(self, agent: Agent):
        turns = 0
        while turns < 9:
            self.board_state.display()
            if turns % 2 == 0:
                print("X's turn")
                self.board_state.place("X", self.ask_mark_position())
                if self.check_game_over("X"):
                    print()
                    print("X wins!")
                    self.board_state.display()
                    break
            else:
                print("O's turn")
                # Extract the agent's best play with epsilon=0
                placement_index = agent.place_mark(self.board_state, 0.0)
                print(f"Agent places O at position {placement_index + 1}")
                self.board_state.place("O", placement_index)
                if self.check_game_over("O"):
                    print()
                    print("O wins!")
                    self.board_state.display()
                    break
            turns += 1
            print()

        if turns == 9:
            print("Draw!")
            self.board_state.display()

    # The game for training, with the agent vs random choices, without prints
    def play_for_training(self, agent: Agent, epsilon: float):
        turns = 0
        # Initialize before and after states needed to update Q properly
        state_before = []
        state_after = []
        placement_index_agent_before = 0
        placement_index_agent_after = 0
        current_reward = 0
        while turns < 9:
            # Set reward for move, overwritten if game ends
            current_reward = agent.reward_move

            if turns % 2 == 0:
                # X, random choice
                placement_index_random = agent.place_mark_random(self.board_state)
                self.board_state.place("X", placement_index_random)
                if self.check_game_over("X"):
                    current_reward = agent.reward_loss
                    break
            else:
                # O, agent
                # If this is the first move, meaning state_before (and state_after) is [], we set state_before;
                # else state_before stays what is was and we update state_after
                # These are always the board states after the opponent has moved
                # Something similar needs to be done for the move the agent makes, so that it matches state_before
                if not state_before:
                    state_before = self.board_state.board.copy()
                else:
                    state_after = self.board_state.board.copy()
                    placement_index_agent_before = placement_index_agent_after
                
                placement_index_agent_after = agent.place_mark(self.board_state, epsilon)
                self.board_state.place("O", placement_index_agent_after)
                
                # Update Q-table with move reward, if it is not the first move, so state_after is not [] anymore;
                # afterwards state_before can be updated to the current state_after for the next iteration
                if state_after:
                    agent.Q_update(state_before, placement_index_agent_before, current_reward, state_after)
                    state_before = state_after

                # Note it is important to put this check after the Q update,
                # to get the correct final update
                if self.check_game_over("O"):
                    current_reward = agent.reward_win
                    break
            
            turns += 1

        if turns == 9:
            current_reward = agent.reward_draw

        # Update Q-table with the final result to reflect a win/loss/draw;
        # in each case state_after still needs to be updated (for a draw the opponent moves last)
        # We can just use the agents move after, which is the last one done and has not been used yet
        state_after = self.board_state.board.copy()
        agent.Q_update(state_before, placement_index_agent_after, current_reward, state_after)

In [ ]:
# Test things out
agent = Agent()
agent.run()

Now starting a game
  |   |  
-- --- --
  |   |  
-- --- --
  |   |  
X's turn



  |   |  
-- --- --
  |   |  
-- --- --
X |   |  
O's turn
Agent places O at position 1

O |   |  
-- --- --
  |   |  
-- --- --
X |   |  
X's turn
Please choose an empty position.

O |   |  
-- --- --
  | X |  
-- --- --
X |   |  
O's turn
Agent places O at position 3

O |   | O
-- --- --
  | X |  
-- --- --
X |   |  
X's turn

O | X | O
-- --- --
  | X |  
-- --- --
X |   |  
O's turn
Agent places O at position 8

O | X | O
-- --- --
  | X |  
-- --- --
X | O |  
X's turn
Please choose an empty position.

O | X | O
-- --- --
  | X |  
-- --- --
X | O | X
O's turn
Agent places O at position 4

O | X | O
-- --- --
O | X |  
-- --- --
X | O | X
X's turn
Please enter an integer.

Draw!
O | X | O
-- --- --
O | X | X
-- --- --
X | O | X


Playing around with this shows the agent knows how to play the game fairly well after 5000 training rounds. Most of the time it will block you from getting 3 in a row and it also recognizes when it can get 3 in a row itself. Still, you will also run into cases where the agent makes suboptimal plays, for instance failing to stop you from winning when it could have. So there is still room for improvement.

One thing we can try is to no longer use a random training opponent, but another agent. Since an agent should, after getting better itself, act more like an actual player, this should help both agents to play better against humans. Now I'm talking about "two agents", but really we only need to define one and have it play against itself. The same Q-table we have filled before can also contain the plays for the "opponent agent", since the relevant input states are disconnected from those of the "original agent", so that these do not interfere with one another.

In [ ]:
# Define the game itself with the training game function
class Game:
    def __init__(self):
        self.board_state = Board()

    def ask_mark_position(self) -> int:
        while True:
            mark_index = input("Choose in which position (1-9) to place your mark: ")
            if not mark_index.isdigit():
                print("Please enter an integer.")
                continue
            
            mark_index = int(mark_index)
            if mark_index < 1 or mark_index > 9:
                print("Please enter a number between 1 and 9.")
            elif self.board_state.board[mark_index - 1] != " ":
                print("Please choose an empty position.")
            else:
                break
        return mark_index - 1

    def check_game_over(self, mark: str) -> bool:
        win_combinations = [
            (0,1,2), (3,4,5), (6,7,8),
            (0,3,6), (1,4,7), (2,5,8),
            (0,4,8), (2,4,6)
        ]
        mark_indexes = np.where(np.array(self.board_state.board, dtype=str) == mark)[0]
        for combination in win_combinations:
            if combination[0] in mark_indexes and combination[1] in mark_indexes and combination[2] in mark_indexes:
                return True
        return False
    
    # Play for training, with agent playing itself
    def play_for_training_agent_vs_agent(self, agent: Agent, epsilon: float):
        turns = 0
        # Initialize before and after states needed to update Q properly
        state_before_O = []
        state_after_O = []
        state_before_X = []
        state_after_X = []
        placement_index_agent_before_O = 0
        placement_index_agent_after_O = 0
        placement_index_agent_before_X = 0
        placement_index_agent_after_X = 0
        # Set reward for move, overwritten if game ends
        current_reward_O = agent.reward_move
        current_reward_X = agent.reward_move
        while turns < 9:
            if turns % 2 == 0:
                # X, "opponent" agent
                # Following the same logic as O now
                if not state_before_X:
                    state_before_X = self.board_state.board.copy()
                else:
                    state_after_X = self.board_state.board.copy()
                    placement_index_agent_before_X = placement_index_agent_after_X

                placement_index_agent_after_X = agent.place_mark(self.board_state, epsilon)
                self.board_state.place("X", placement_index_agent_after_X)

                if state_after_X:
                    agent.Q_update(state_before_X, placement_index_agent_before_X, current_reward_X, state_after_X)
                    state_before_X = state_after_X
                
                if self.check_game_over("X"):
                    current_reward_O = agent.reward_loss
                    current_reward_X = agent.reward_win
                    break
            else:
                # O, agent
                # If this is the first move, meaning state_before (and state_after) is [], we set state_before;
                # else state_before stays what is was and we update state_after
                # These are always the board states after the opponent has moved
                # Something similar needs to be done for the move the agent makes, so that it matches state_before
                if not state_before_O:
                    state_before_O = self.board_state.board.copy()
                else:
                    state_after_O = self.board_state.board.copy()
                    placement_index_agent_before_O = placement_index_agent_after_O
                
                placement_index_agent_after_O = agent.place_mark(self.board_state, epsilon)
                self.board_state.place("O", placement_index_agent_after_O)
                
                # Update Q-table with move reward, if it is not the first move, so state_after is not [] anymore;
                # afterwards state_before can be updated to the current state_after for the next iteration
                if state_after_O:
                    agent.Q_update(state_before_O, placement_index_agent_before_O, current_reward_O, state_after_O)
                    state_before_O = state_after_O

                # Note it is important to put this check after the Q update,
                # to get the correct final update
                if self.check_game_over("O"):
                    current_reward_O = agent.reward_win
                    current_reward_X = agent.reward_loss
                    break
            
            turns += 1

        if turns == 9:
            current_reward_O = agent.reward_draw
            current_reward_X = agent.reward_draw

        # Update Q-table with the final result to reflect a win/loss/draw;
        # in each case state_after still needs to be updated (for a draw the opponent moves last)
        # We can just use the agents move after, which is the last one done and has not been used yet
        state_after_O = self.board_state.board.copy()
        agent.Q_update(state_before_O, placement_index_agent_after_O, current_reward_O, state_after_O)
        state_after_X = self.board_state.board.copy()
        agent.Q_update(state_before_X, placement_index_agent_after_X, current_reward_X, state_after_X)

Interestingly enough, testing the agent that is trained like this (which was done in the .py file as it is easier to interact with the terminal there) seems to lead to worse results, as we run into bad plays more frequently. This likely happens because as both agents are trained and $\epsilon$ decreases, they keep repeating the same moves that they have learned are good so far, which results in them exploring less states than before. This leads to them not being able to make good plays for many unknown board states, and possibly also to them getting somewhat stuck in the same moves despite the $\epsilon$ method.

(Note: from now on, further changes and additions to the code are only applied in the .py file, as it is much easier to run the script there and interact with the terminal. Please consult that file to see the code related to the results and explanations below.)

Adding a `test` function to the `Agent` class will let us examine how well the agent performs after it has been trained. Here we let it play against a fully random opponent, for 10,000 rounds. Although this is not the same as playing against a human, it does give us a reliable way to test how well any type of opponent, as the randomness should lead to both good and bad plays on that side. The results of our first two training modes are the following:

Mode 1, against random opponent:
- Results: 83.59% wins, 9.54% draws, 6.87% losses
- Results: 83.19% wins, 10.28% draws, 6.53% losses

Mode 2, against simultaneously learning agent:
- Results: 57.03% wins, 11.86% draws, 31.11% losses
- Results: 57.25% wins, 12.26% draws, 30.49% losses

Clearly, the agent performs quite a bit better when trained against a fully random opponent.

## Pretrained agent training

Now we can also try to use a "pretrained" agent instead, i.e. one that is not trained simultaneously with the original agent, but before it starts its own training. This is implemented in the `play_for_training_vs_pretrained_agent` function. Note that the pretrained agent needs to be trained to play as X rather than O, so we also add the `play_for_training_vs_random_X` function to do this. This leads to the following results:

Mode 3, against pretrained agent:
- Results: 30.18% wins, 13.15% draws, 56.67% losses
- Results: 30.10% wins, 11.98% draws, 57.92% losses

This is quite bad. It happens because the problem from before kind of got worse, in the sense that the pretrained agent is never making random moves, but always uses its best learned plays, reducing the amount of different board states that the agent in training gets to see even further.

To fix this, we can actually use the same probability parameter $\epsilon$, but set it to some fixed value, so that the pretrained agent does still make random moves sometimes (denoted in the code by `epsilon_X`). When trying this, you see that the win rate improves as we increase `epsilon_X`. However, there is no better result before we get to `epsilon_X=1.0`, i.e. a fully random opponent again; it keeps constantly increasing. So this pretrained agent likely never gives a better result.

## Combining different $\epsilon$ values

One final thing to try, is that we could combine the training with different values of $\epsilon$. So instead of having the pretrained agent use one value, we play different rounds where it is first completely determined, then partially random, until it is fully random. This way the agent in training gets to see many states, but also better plays than just a random move.

This is implemented by running the `play_for_training_vs_pretrained_agent` function multiple times for each chosen $\epsilon$ value. Note that this way, `num_episodes` is not actually the total number of training rounds anymore, but rather `num_episodes` times the amount of epsilon values (you can always decrease `num_episodes` accordingly if you wish to keep the same number of total rounds).

So trying this with `num_episodes=1000` and $\epsilon$ values of `[1.0, 0.75, 0.5, 0.25, 0.0]`, so that we have 5000 training rounds in total like before, the results are:

Mode 4, against pretrained agent with varying epsilon_X:
- Results: 70.74% wins, 11.22% draws, 18.04% losses
- Results: 70.29% wins, 11.15% draws, 18.56% losses

Although this is getting better, it is still not at the level of a purely random training opponent. Presumably it is still better to see many different random board configurations than having a better opponent sometimes.

## More training

The most straightforward way to improve results further, although it is computationally expensive, is to increase the number of training rounds. Using the random training opponent, this gets the following results:

25k rounds:
- Results: 84.96% wins, 10.66% draws, 4.38% losses
- Results: 85.33% wins, 10.32% draws, 4.35% losses

50k rounds:
- Results: 85.77% wins, 11.27% draws, 2.96% losses
- Results: 86.22% wins, 11.30% draws, 2.48% losses

100k rounds:
- Results: 91.46% wins, 7.71% draws, 0.83% losses
- Results: 91.83% wins, 7.27% draws, 0.90% losses

500k rounds:
- Results: 91.76% wins, 7.55% draws, 0.69% losses
- Results: 91.90% wins, 7.38% draws, 0.72% losses

Slowly but surely, you can see the win rate increase. I'm not sure why there is such a jump between 50k and 100k rounds though. We can conclude that more random training leads to better results than a smarter opponent.

One thing I noticed is that for 100k training rounds or more, the agent starts to consistently take the middle spot (5) on its first turn if we leave it open on our first turn. This is probably the best strategy there is, as I have noticed it becomes impossible to beat it at that point if it continues to play well. Also, in case we do start in the middle, it has learned to take one of the diagonal spots (1, 3, 7, 9), which is the best way to guarantee at least a draw if it plays well.

In general, the agent after 500k training rounds plays the game very well. During my test games against it, I did not see it make any "silly" plays, where it e.g. does not block you while it could have. It is therefore really difficult to beat. One way I did find is by placing a mark on position 5, after which the agent places one at 9, then we place one at 1, and then it places one at 6. This does not seem like a mistake at first, but after we block the agent by placing one at 3, there are two ways for use to get three in a row (2 and 7), so that we have a guaranteed win. The agent should have placed its mark on a diagonal instead, but since this is one of the more complex situations you can get in this game, it is probably hard to learn for it. (Of course, if it were to be trained again, this might not apply anymore due to the randomness, and you will have to find some other way to beat it.) Perhaps the agent can get completely unbeatable if you train it even longer, but 500k training rounds already take a while on my machine so I won't be exploring that further.

Now we could also try to give the pretrained agent more training rounds to maybe get better results for that method, but I doubt it will end up leading to better results than the purely random opponent. It also becomes very computationally expensive if we have to train both the original and the opponent agent for many rounds. Given that the purely random opponent training also gives great results, we will leave it at that.